In [15]:
!pip install ultralytics -q

In [16]:
from ultralytics import YOLO
import os
import yaml

In [17]:
import os

base_path = "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data"

print(os.listdir(base_path))

['README.dataset.txt', 'README.roboflow.txt', 'valid', 'test', 'train']


In [18]:
%%writefile /kaggle/working/data.yaml

path: /kaggle/input/construction-site-safety-image-dataset-roboflow/css-data
train: train/images
val: valid/images
test: test/images

names:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest

Overwriting /kaggle/working/data.yaml


In [20]:
%%writefile /kaggle/working/aa_yolo.yaml

nc: 5

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, SPPF, [256, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [64]]

  - [-1, 1, Conv, [64, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [[13, 16, 19], 1, Detect, [nc]]

Overwriting /kaggle/working/aa_yolo.yaml


In [21]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/aa_yolo.yaml')

print('AA-YOLO Loaded Successfully')

AA-YOLO Loaded Successfully


In [22]:
import os

base_path = "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data"

for folder in os.listdir(base_path):
    print(folder)

README.dataset.txt
README.roboflow.txt
valid
test
train


In [23]:
%%writefile /kaggle/working/data.yaml

path: /kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data
train: train/images
val: valid/images
test: test/images

names:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest

Overwriting /kaggle/working/data.yaml


In [24]:
!cat /kaggle/working/data.yaml


path: /kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data
train: train/images
val: valid/images
test: test/images

names:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest


In [25]:
%%writefile /kaggle/working/aa_yolo.yaml

nc: 5

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, SPPF, [256, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [64]]

  - [-1, 1, Conv, [64, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [[13, 16, 19], 1, Detect, [nc]]

Overwriting /kaggle/working/aa_yolo.yaml


In [26]:
import os
import shutil
from tqdm import tqdm

# Original dataset path
SOURCE = "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data"

# Filtered dataset output
DEST = "/kaggle/working/css_filtered"

# Classes we keep
allowed_classes = [0, 2, 5, 7, 4]

# Remap to 0–4
class_map = {
    0: 0,   # Hardhat
    2: 1,   # NO-Hardhat
    5: 2,   # Person
    7: 3,   # Safety Vest
    4: 4    # NO-Safety Vest
}

splits = ["train", "valid", "test"]

for split in splits:

    os.makedirs(f"{DEST}/{split}/images", exist_ok=True)
    os.makedirs(f"{DEST}/{split}/labels", exist_ok=True)

    label_dir = f"{SOURCE}/{split}/labels"
    image_dir = f"{SOURCE}/{split}/images"

    for label_file in tqdm(os.listdir(label_dir)):

        label_path = os.path.join(label_dir, label_file)

        new_lines = []

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()

            cls = int(parts[0])

            if cls in allowed_classes:
                parts[0] = str(class_map[cls])
                new_lines.append(" ".join(parts))

        if len(new_lines) > 0:

            image_file = label_file.replace(".txt", ".jpg")

            src_img = os.path.join(image_dir, image_file)
            dst_img = os.path.join(f"{DEST}/{split}/images", image_file)

            src_lbl = label_path
            dst_lbl = os.path.join(f"{DEST}/{split}/labels", label_file)

            if os.path.exists(src_img):
                shutil.copy(src_img, dst_img)

                with open(dst_lbl, "w") as f:
                    f.write("\n".join(new_lines))

print("Filtered dataset created successfully.")

100%|██████████| 82/82 [00:00<00:00, 320.15it/s]

Filtered dataset created successfully.


In [27]:
import os

for split in ["train", "valid", "test"]:
    img_count = len(os.listdir(f"/kaggle/working/css_filtered/{split}/images"))
    lbl_count = len(os.listdir(f"/kaggle/working/css_filtered/{split}/labels"))

    print(split)
    print("Images:", img_count)
    print("Labels:", lbl_count)
    print()

train
Images: 2535
Labels: 2535

valid
Images: 84
Labels: 84

test
Images: 60
Labels: 60



In [28]:
%%writefile /kaggle/working/data.yaml

train: /kaggle/working/css_filtered/train/images
val: /kaggle/working/css_filtered/valid/images
test: /kaggle/working/css_filtered/test/images

nc: 5

names:
  0: Hardhat
  1: NO-Hardhat
  2: Person
  3: Safety Vest
  4: NO-Safety Vest

Overwriting /kaggle/working/data.yaml


In [29]:
!cat /kaggle/working/data.yaml


train: /kaggle/working/css_filtered/train/images
val: /kaggle/working/css_filtered/valid/images
test: /kaggle/working/css_filtered/test/images

nc: 5

names:
  0: Hardhat
  1: NO-Hardhat
  2: Person
  3: Safety Vest
  4: NO-Safety Vest


In [30]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/aa_yolo.yaml')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=120,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    project='/kaggle/working/runs',
    name='AA_YOLO_v4'
)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/aa_yolo.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=AA_YOLO_v4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

In [36]:
!pip install -q ultralytics

from ultralytics import YOLO

In [37]:
model = YOLO("yolov8m.pt")

In [39]:
!cat /kaggle/working/data.yaml


train: /kaggle/working/css_filtered/train/images
val: /kaggle/working/css_filtered/valid/images
test: /kaggle/working/css_filtered/test/images

nc: 5

names:
  0: Hardhat
  1: NO-Hardhat
  2: Person
  3: Safety Vest
  4: NO-Safety Vest


In [40]:
model.train(
    data="/kaggle/working/data.yaml",
    imgsz=832,

    epochs=120,
    batch=16,
    device=0,

    optimizer="AdamW",
    lr0=0.001,


    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.25,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    close_mosaic=10,

    patience=30
)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.25, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, pe

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78798c1497f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

**"For Later Versions"**# 

In [1]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.6 MB/s eta 0:00:0000:01


In [2]:
from ultralytics import YOLO
import os
import yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import os

base_path = "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data"

print(os.listdir(base_path))

['README.dataset.txt', 'README.roboflow.txt', 'valid', 'test', 'train']


In [4]:
%%writefile /kaggle/working/data.yaml

path: /kaggle/input/construction-site-safety-image-dataset-roboflow/css-data
train: train/images
val: valid/images
test: test/images

names:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest

Writing /kaggle/working/data.yaml


In [5]:
import os

base_path = "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data"

for folder in os.listdir(base_path):
    print(folder)

README.dataset.txt
README.roboflow.txt
valid
test
train


In [6]:
%%writefile /kaggle/working/data.yaml

path: /kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data
train: train/images
val: valid/images
test: test/images

names:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest

Overwriting /kaggle/working/data.yaml


In [7]:
!cat /kaggle/working/data.yaml


path: /kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data
train: train/images
val: valid/images
test: test/images

names:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest


In [9]:
%%writefile /kaggle/working/aa_yolo.yaml

nc: 5

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, SPPF, [256, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [64]]

  - [-1, 1, Conv, [64, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [[13, 16, 19], 1, Detect, [nc]]

Overwriting /kaggle/working/aa_yolo.yaml


In [10]:
import os
import shutil
from tqdm import tqdm

# Original dataset path
SOURCE = "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data"

# Filtered dataset output
DEST = "/kaggle/working/css_filtered"

# Classes we keep
allowed_classes = [0, 2, 5, 7, 4]

# Remap to 0–4
class_map = {
    0: 0,   # Hardhat
    2: 1,   # NO-Hardhat
    5: 2,   # Person
    7: 3,   # Safety Vest
    4: 4    # NO-Safety Vest
}

splits = ["train", "valid", "test"]

for split in splits:

    os.makedirs(f"{DEST}/{split}/images", exist_ok=True)
    os.makedirs(f"{DEST}/{split}/labels", exist_ok=True)

    label_dir = f"{SOURCE}/{split}/labels"
    image_dir = f"{SOURCE}/{split}/images"

    for label_file in tqdm(os.listdir(label_dir)):

        label_path = os.path.join(label_dir, label_file)

        new_lines = []

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()

            cls = int(parts[0])

            if cls in allowed_classes:
                parts[0] = str(class_map[cls])
                new_lines.append(" ".join(parts))

        if len(new_lines) > 0:

            image_file = label_file.replace(".txt", ".jpg")

            src_img = os.path.join(image_dir, image_file)
            dst_img = os.path.join(f"{DEST}/{split}/images", image_file)

            src_lbl = label_path
            dst_lbl = os.path.join(f"{DEST}/{split}/labels", label_file)

            if os.path.exists(src_img):
                shutil.copy(src_img, dst_img)

                with open(dst_lbl, "w") as f:
                    f.write("\n".join(new_lines))

print("Filtered dataset created successfully.")

100%|██████████| 82/82 [00:00<00:00, 84.09it/s]

Filtered dataset created successfully.


In [11]:
import os

for split in ["train", "valid", "test"]:
    img_count = len(os.listdir(f"/kaggle/working/css_filtered/{split}/images"))
    lbl_count = len(os.listdir(f"/kaggle/working/css_filtered/{split}/labels"))

    print(split)
    print("Images:", img_count)
    print("Labels:", lbl_count)
    print()

train
Images: 2535
Labels: 2535

valid
Images: 84
Labels: 84

test
Images: 60
Labels: 60



In [12]:
%%writefile /kaggle/working/data.yaml

train: /kaggle/working/css_filtered/train/images
val: /kaggle/working/css_filtered/valid/images
test: /kaggle/working/css_filtered/test/images

nc: 5

names:
  0: Hardhat
  1: NO-Hardhat
  2: Person
  3: Safety Vest
  4: NO-Safety Vest

Overwriting /kaggle/working/data.yaml


In [13]:
!cat /kaggle/working/data.yaml


train: /kaggle/working/css_filtered/train/images
val: /kaggle/working/css_filtered/valid/images
test: /kaggle/working/css_filtered/test/images

nc: 5

names:
  0: Hardhat
  1: NO-Hardhat
  2: Person
  3: Safety Vest
  4: NO-Safety Vest


In [14]:
from ultralytics import YOLO
import shutil
import os

model = YOLO('/kaggle/working/aa_yolo.yaml')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,

    project='/kaggle/working/runs',
    name='AA_YOLO_v2',
    exist_ok=True,

    save=True,
    save_period=1,
    plots=True,
    verbose=True
)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/aa_yolo.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=AA_YOLO_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tru

In [32]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/AA_YOLO_v2/weights/best.pt')

model.predict(
    source='/kaggle/working/css_filtered/test/images',
    save=True,
    conf=0.25
)


image 1/60 /kaggle/working/css_filtered/test/images/-4405-_png_jpg.rf.82b5c10b2acd1cfaa24259ada8e599fe.jpg: 640x640 1 Hardhat, 1 Person, 1 NO-Safety Vest, 20.8ms
image 2/60 /kaggle/working/css_filtered/test/images/000005_jpg.rf.96e9379ccae638140c4a90fc4b700a2b.jpg: 640x640 2 Hardhats, 3 Persons, 20.8ms
image 3/60 /kaggle/working/css_filtered/test/images/002551_jpg.rf.ce4b9f934161faa72c80dc6898d37b2d.jpg: 640x640 2 Hardhats, 3 Persons, 2 NO-Safety Vests, 20.8ms
image 4/60 /kaggle/working/css_filtered/test/images/003357_jpg.rf.9867f91e88089bb68dc95947d5116d14.jpg: 640x640 1 Hardhat, 2 Persons, 20.8ms
image 5/60 /kaggle/working/css_filtered/test/images/004063_jpg.rf.1b7cdc4035bcb24ef69b8798b444053e.jpg: 640x640 5 Hardhats, 1 NO-Hardhat, 6 Persons, 1 Safety Vest, 5 NO-Safety Vests, 20.8ms
image 6/60 /kaggle/working/css_filtered/test/images/004763_jpg.rf.46484e6ca73caeaa9de45822cf1085a9.jpg: 640x640 3 Hardhats, 4 Persons, 4 NO-Safety Vests, 20.8ms
image 7/60 /kaggle/working/css_filtered/te

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Hardhat', 1: 'NO-Hardhat', 2: 'Person', 3: 'Safety Vest', 4: 'NO-Safety Vest'}
 obb: None
 orig_img: array([[[ 81, 113, 124],
         [ 75, 105, 116],
         [ 66,  94, 105],
         ...,
         [ 64,  60,  72],
         [ 58,  54,  66],
         [ 54,  50,  62]],
 
        [[ 77, 109, 120],
         [ 73, 103, 114],
         [ 67,  95, 106],
         ...,
         [ 58,  54,  66],
         [ 54,  50,  62],
         [ 52,  48,  60]],
 
        [[ 81, 113, 124],
         [ 81, 111, 122],
         [ 79, 108, 117],
         ...,
         [ 44,  42,  54],
         [ 45,  41,  53],
         [ 44,  40,  52]],
 
        ...,
 
        [[ 20,  20,  20],
         [ 22,  22,  22],
         [ 23,  23,  23],
         ...,
         [ 62,  66,  67],
         [ 61,  66,  69],
         [ 61,  66,  69]],
 
        [[ 19,  19,  19],
         [ 20,

In [33]:
aa_no_attention_yaml = """
nc: 5

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 2, C2f, [64, True]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 3, C2f, [256, True]]
  - [-1, 1, SPPF, [256, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [64]]

  - [-1, 1, Conv, [64, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [[13, 16, 19], 1, Detect, [nc]]
"""

with open("/kaggle/working/aa_yolo_no_attention.yaml", "w") as f:
    f.write(aa_no_attention_yaml)

print("Saved")

Saved


In [34]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/aa_yolo_no_attention.yaml')

model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    name='AA_YOLO_NoAttention'
)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/aa_yolo_no_attention.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=AA_YOLO_NoAttention, nbs=64, nms=False, opset=None, optimize=False, optimizer

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7df87f097140>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [35]:
import time
from ultralytics import YOLO
import cv2
import os

model = YOLO('/kaggle/working/runs/AA_YOLO_v2/weights/best.pt')

image_folder = "/kaggle/working/css_filtered/test/images"

image_paths = [
    os.path.join(image_folder, f)
    for f in os.listdir(image_folder)
][:100]

start = time.time()

for img in image_paths:
    model.predict(img, verbose=False)

end = time.time()

total_time = end - start
fps = len(image_paths) / total_time

print(f"FPS: {fps:.2f}")
print(f"Inference Time/Image: {1000/fps:.2f} ms")

FPS: 49.24
Inference Time/Image: 20.31 ms


In [36]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/AA_YOLO_v2/weights/best.pt')

model.info()

aa_YOLO summary: 164 layers, 4,765,119 parameters, 0 gradients, 41.7 GFLOPs


(164, 4765119, 0, 41.68048640000001)

In [37]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO('/kaggle/working/runs/AA_YOLO_v2/weights/best.pt')

results = model.predict(
    source='/kaggle/working/css_filtered/test/images',
    save=True,
    conf=0.25
)

print("Saved attention predictions")


image 1/60 /kaggle/working/css_filtered/test/images/-4405-_png_jpg.rf.82b5c10b2acd1cfaa24259ada8e599fe.jpg: 640x640 1 Hardhat, 1 Person, 1 NO-Safety Vest, 20.8ms
image 2/60 /kaggle/working/css_filtered/test/images/000005_jpg.rf.96e9379ccae638140c4a90fc4b700a2b.jpg: 640x640 2 Hardhats, 3 Persons, 20.8ms
image 3/60 /kaggle/working/css_filtered/test/images/002551_jpg.rf.ce4b9f934161faa72c80dc6898d37b2d.jpg: 640x640 2 Hardhats, 3 Persons, 2 NO-Safety Vests, 20.8ms
image 4/60 /kaggle/working/css_filtered/test/images/003357_jpg.rf.9867f91e88089bb68dc95947d5116d14.jpg: 640x640 1 Hardhat, 2 Persons, 20.7ms
image 5/60 /kaggle/working/css_filtered/test/images/004063_jpg.rf.1b7cdc4035bcb24ef69b8798b444053e.jpg: 640x640 5 Hardhats, 1 NO-Hardhat, 6 Persons, 1 Safety Vest, 5 NO-Safety Vests, 20.2ms
image 6/60 /kaggle/working/css_filtered/test/images/004763_jpg.rf.46484e6ca73caeaa9de45822cf1085a9.jpg: 640x640 3 Hardhats, 4 Persons, 4 NO-Safety Vests, 15.4ms
image 7/60 /kaggle/working/css_filtered/te

In [38]:
models = {
    "YOLOv8n": [3.0, 8.1],
    "YOLOv8s": [11.1, 28.4],
    "YOLOv8m": [25.8, 78.7],
    "AA-YOLO": [4.76, 41.3]
}

print("Model | Params(M) | GFLOPs")
print("-"*35)

for k,v in models.items():
    print(f"{k} | {v[0]} | {v[1]}")

Model | Params(M) | GFLOPs
-----------------------------------
YOLOv8n | 3.0 | 8.1
YOLOv8s | 11.1 | 28.4
YOLOv8m | 25.8 | 78.7
AA-YOLO | 4.76 | 41.3
